# Data Preprocessing in TuiML

This tutorial covers the 60+ preprocessing transformers available in TuiML for data preparation, feature engineering, and handling imbalanced datasets.

In [1]:
# Common imports
import numpy as np
from tuiml.datasets import load_iris, load_diabetes
from tuiml.evaluation import train_test_split, accuracy_score

## 1. Numerical Scaling

Scale numerical features to improve model performance.

In [2]:
from tuiml.preprocessing import MinMaxScaler, StandardScaler, CenterScaler

X, y = load_iris()

print("Original data:")
print(f"  Min: {X.min(axis=0)}")
print(f"  Max: {X.max(axis=0)}")
print(f"  Mean: {X.mean(axis=0)}")
print(f"  Std: {X.std(axis=0)}")

Original data:
  Min: [4.3 2.  1.  0.1]
  Max: [7.9 4.4 6.9 2.5]
  Mean: [5.84333333 3.054      3.75866667 1.19866667]
  Std: [0.82530129 0.43214658 1.75852918 0.76061262]


In [3]:
# MinMaxScaler (Min-Max scaling to [0, 1])
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X)
print("MinMaxScaler (scaled to [0,1]):")
print(f"  Min: {X_norm.min(axis=0)}")
print(f"  Max: {X_norm.max(axis=0)}")

MinMaxScaler (scaled to [0,1]):
  Min: [0. 0. 0. 0.]
  Max: [1. 1. 1. 1.]


In [4]:
# StandardScaler (Z-score normalization)
standardizer = StandardScaler()
X_std = standardizer.fit_transform(X)
print("StandardScaler (Z-score):")
print(f"  Mean: {X_std.mean(axis=0).round(10)}")
print(f"  Std: {X_std.std(axis=0)}")

StandardScaler (Z-score):
  Mean: [-0. -0.  0.  0.]
  Std: [1. 1. 1. 1.]


In [5]:
# CenterScaler (subtract mean only)
centerer = CenterScaler()
X_centered = centerer.fit_transform(X)
print("CenterScaler (mean subtracted):")
print(f"  Mean: {X_centered.mean(axis=0).round(10)}")
print(f"  Std: {X_centered.std(axis=0)}")

CenterScaler (mean subtracted):
  Mean: [-0. -0.  0.  0.]
  Std: [0.82530129 0.43214658 1.75852918 0.76061262]


## 2. Missing Value Imputation

Handle missing values in your data.

In [6]:
from tuiml.preprocessing import SimpleImputer, KNNImputer

# Create data with missing values
X_missing = X.copy().astype(float)
np.random.seed(42)
mask = np.random.random(X_missing.shape) < 0.1
X_missing[mask] = np.nan

print(f"Missing values: {np.isnan(X_missing).sum()}")

Missing values: 72


In [7]:
# Simple Imputer with mean strategy
imputer_mean = SimpleImputer(strategy='mean')
X_imputed_mean = imputer_mean.fit_transform(X_missing)
print(f"After mean imputation - Missing: {np.isnan(X_imputed_mean).sum()}")

# Simple Imputer with median strategy
imputer_median = SimpleImputer(strategy='median')
X_imputed_median = imputer_median.fit_transform(X_missing)
print(f"After median imputation - Missing: {np.isnan(X_imputed_median).sum()}")

After mean imputation - Missing: 0
After median imputation - Missing: 0


In [8]:
# KNN Imputer (uses nearest neighbors)
knn_imputer = KNNImputer(n_neighbors=5)
X_imputed_knn = knn_imputer.fit_transform(X_missing)
print(f"After KNN imputation - Missing: {np.isnan(X_imputed_knn).sum()}")

After KNN imputation - Missing: 0


## 3. Discretization

Convert continuous features to categorical bins.

In [9]:
from tuiml.preprocessing import EqualWidthDiscretizer, QuantileDiscretizer

# Equal Width Discretization
ew_disc = EqualWidthDiscretizer(n_bins=5)
X_ew = ew_disc.fit_transform(X[:, 0:1])  # Apply to first feature
print("Equal Width Discretization (5 bins):")
print(f"  Unique values: {np.unique(X_ew)}")
print(f"  Value counts: {np.bincount(X_ew.astype(int).flatten())}")

Equal Width Discretization (5 bins):
  Unique values: [0. 1. 2. 3. 4.]
  Value counts: [32 41 42 24 11]


In [10]:
# Quantile Discretization (equal frequency bins)
ef_disc = QuantileDiscretizer(n_bins=5)
X_ef = ef_disc.fit_transform(X[:, 0:1])
print("Quantile Discretization (5 bins):")
print(f"  Unique values: {np.unique(X_ef)}")
print(f"  Value counts: {np.bincount(X_ef.astype(int).flatten())}")

Quantile Discretization (5 bins):
  Unique values: [0. 1. 2. 3. 4.]
  Value counts: [22 37 30 31 30]


## 4. Outlier Handling

In [11]:
from tuiml.preprocessing import IQROutlierDetector, ValueClipper

# Add some outliers
X_with_outliers = X.copy()
X_with_outliers[0, 0] = 100  # Extreme outlier
X_with_outliers[1, 0] = -50

print(f"Original range: [{X_with_outliers[:, 0].min()}, {X_with_outliers[:, 0].max()}]")

Original range: [-50.0, 100.0]


In [12]:
# Clip values to percentile range
clipper = ValueClipper(percentile=(5, 95))
X_clipped = clipper.fit_transform(X_with_outliers)
print(f"After clipping: [{X_clipped[:, 0].min():.2f}, {X_clipped[:, 0].max():.2f}]")

After clipping: [4.60, 7.35]


In [13]:
# IQR-based outlier detection
iqr_handler = IQROutlierDetector(factor=1.5)
X_iqr = iqr_handler.fit_transform(X_with_outliers)
print(f"After IQR handling: [{X_iqr[:, 0].min():.2f}, {X_iqr[:, 0].max():.2f}]")

After IQR handling: [3.15, 8.35]


## 5. Sampling and Imbalanced Learning

Handle imbalanced datasets with various sampling strategies.

In [14]:
from tuiml.preprocessing import (
    SMOTESampler, 
    RandomOverSampler, 
    RandomUnderSampler,
    ClassBalanceSampler
)

# Create imbalanced dataset
X, y = load_iris()
# Make it imbalanced by keeping only some samples of class 0
mask = (y != 0) | (np.random.random(len(y)) < 0.2)
X_imb, y_imb = X[mask], y[mask]

print("Imbalanced dataset:")
for c in np.unique(y_imb):
    print(f"  Class {c}: {(y_imb == c).sum()} samples")

Imbalanced dataset:
  Class 0: 12 samples
  Class 1: 50 samples
  Class 2: 50 samples


In [15]:
# SMOTESampler (Synthetic Minority Over-sampling Technique)
smote = SMOTESampler(sampling_strategy='auto', random_state=42)
X_smote, y_smote = smote.fit_resample(X_imb, y_imb)

print("\nAfter SMOTESampler:")
for c in np.unique(y_smote):
    print(f"  Class {c}: {(y_smote == c).sum()} samples")


After SMOTESampler:
  Class 0: 50 samples
  Class 1: 50 samples
  Class 2: 50 samples


In [16]:
# Random Under-sampling (reduce majority class)
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_imb, y_imb)

print("\nAfter Random Under-sampling:")
for c in np.unique(y_rus):
    print(f"  Class {c}: {(y_rus == c).sum()} samples")


After Random Under-sampling:
  Class 0: 12 samples
  Class 1: 12 samples
  Class 2: 12 samples


## 6. Text Preprocessing

Transform text data into numerical features.

In [17]:
from tuiml.preprocessing import (
    TfidfVectorizer, 
    CountVectorizer,
    TextCleaner,
    StopWordRemover
)

# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing deals with text and speech.",
    "Data science combines statistics and machine learning.",
    "AI and ML are transforming many industries today."
]

In [18]:
# Count Vectorizer (Bag of Words)
count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(documents)
print("Count Vectorizer:")
print(f"  Shape: {X_counts.shape}")
print(f"  Vocabulary size: {len(count_vec.vocabulary_)}")
print(f"  First 10 terms: {list(count_vec.vocabulary_.keys())[:10]}")

Count Vectorizer:
  Shape: (5, 32)
  Vocabulary size: 32
  First 10 terms: ['and', 'learning', 'machine', 'many', 'with', 'a', 'ai', 'are', 'artificial', 'combines']


In [19]:
# TF-IDF Vectorizer
tfidf_vec = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf_vec.fit_transform(documents)
print("TF-IDF Vectorizer:")
print(f"  Shape: {X_tfidf.shape}")
print(f"  First document vector (non-zero entries):")
non_zero = np.nonzero(X_tfidf[0])[0]
vocab_items = list(tfidf_vec.vocabulary_.items())
for idx in non_zero[:5]:
    term = [k for k, v in vocab_items if v == idx][0]
    print(f"    {term}: {X_tfidf[0, idx]:.4f}")

TF-IDF Vectorizer:
  Shape: (5, 26)
  First document vector (non-zero entries):
    learning: 0.3308
    machine: 0.3985
    artificial: 0.4939
    intelligence: 0.4939
    subset: 0.4939


## 7. Attribute Manipulation

In [20]:
from tuiml.preprocessing import AttributeRemover, UselessAttributeRemover, AttributeReorderer

X, y = load_iris()
print(f"Original shape: {X.shape}")

# Remove specific columns
remover = AttributeRemover(columns=[0, 2])  # Remove first and third columns
X_reduced = remover.fit_transform(X)
print(f"After removing columns 0, 2: {X_reduced.shape}")

Original shape: (150, 4)
After removing columns 0, 2: (150, 2)


In [21]:
# Remove useless attributes (constant values, etc.)
X_with_const = np.column_stack([X, np.ones(len(X))])  # Add constant column
print(f"With constant column: {X_with_const.shape}")

useless_remover = UselessAttributeRemover()
X_clean = useless_remover.fit_transform(X_with_const)
print(f"After removing useless: {X_clean.shape}")

With constant column: (150, 5)
After removing useless: (150, 4)


## 8. Time Series Preprocessing

In [22]:
from tuiml.preprocessing import LagTransformer, DifferenceTransformer

# Create time series data
ts_data = np.arange(10).reshape(-1, 1).astype(float)
print("Original time series:")
print(ts_data.flatten())

Original time series:
[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]


In [23]:
# Create lag features
lag_transformer = LagTransformer(lag=-1)
ts_lagged = lag_transformer.fit_transform(ts_data)
print("\nWith lag features:")
print(ts_lagged)


With lag features:
[[nan]
 [ 0.]
 [ 1.]
 [ 2.]
 [ 3.]
 [ 4.]
 [ 5.]
 [ 6.]
 [ 7.]
 [ 8.]]


In [24]:
# Create difference features
delta_transformer = DifferenceTransformer(lag=-1)
ts_delta = delta_transformer.fit_transform(ts_data)
print("\nWith difference features:")
print(ts_delta.flatten())


With difference features:
[nan  1.  1.  1.  1.  1.  1.  1.  1.  1.]


## 9. Preprocessing Pipeline Example

Combine multiple preprocessing steps for a complete workflow.

In [25]:
from tuiml.preprocessing import SimpleImputer, StandardScaler
from tuiml.algorithms.trees import RandomForestClassifier

# Load data
X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Add some missing values to training data
X_train_missing = X_train.copy()
mask = np.random.random(X_train_missing.shape) < 0.05
X_train_missing[mask] = np.nan

print(f"Training data with {np.isnan(X_train_missing).sum()} missing values")

# Step 1: Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_missing)
X_test_imputed = imputer.transform(X_test)  # Use same imputation

# Step 2: StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)  # Use same scaling

# Step 3: Train model
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.2%}")

Training data with 27 missing values

Accuracy: 100.00%
